=============================================================================
STUDENT 5 — TEXT ANALYST (BUILT FROM SCRATCH)
Multimodal Crime / Incident Report Analyzer
AI for Engineers — Group Assignment
=============================================================================
DATASET: CrimeReport — Kaggle
Link: https://www.kaggle.com/datasets/cameliasiadat/crimereport

HOW TO USE:
Step 1 → Go to kaggle.com/datasets/cameliasiadat/crimereport
Step 2 → Sign in → Click Download → saves crimereport.csv to your computer
Step 3 → Upload crimereport.csv to Google Drive:
MyDrive/multimodal_crime_analyzer/text_data/crimereport.csv
Step 4 → Upload THIS notebook to Google Colab
Step 5 → Run All cells

Following assignment tip:
"The dataset is already in CSV format — skip scraping entirely.
Focus on building NER with spaCy, sentiment analysis with HuggingFace,
and topic classification."

STRUCTURE:
PART 1 → Load CSV directly — df = pd.read_csv('crimereport.csv')
PART 2 → Text preprocessing (clean, normalize, tokenize) using NLTK
PART 3 → NER + Sentiment + Topic Classification
=============================================================================

### Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Install All Dependencies

In [2]:
import subprocess
subprocess.run(["pip", "install", "spacy", "transformers",
                "torch", "nltk", "pandas", "-q"])
subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm", "-q"])
print("✔ All dependencies installed")

✔ All dependencies installed


### Imports

In [3]:
import os
import re
import warnings
import pandas as pd
import spacy
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from transformers import pipeline
from IPython.display import display

warnings.filterwarnings("ignore")

# Download NLTK resources
nltk.download("punkt",     quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("punkt_tab", quiet=True)

print("✔ Imports done")

✔ Imports done


### Configuration — SET YOUR PATH HERE

In [4]:
# Path to your crimereport.csv in Google Drive
CSV_PATH   = "/content/drive/MyDrive/AI_DATASETS/TEXT_AI/CrimeReport (1).txt"
TXT_PATH   = CSV_PATH
OUTPUT_CSV = "/content/drive/MyDrive/AI_DATASETS/TEXT_AI.csv"

# Process first 200 rows — enough for a strong prototype
MAX_ROWS = 200

### Load NLP Models

In [5]:
print("Loading spaCy NER model...")
nlp = spacy.load("en_core_web_sm")

print("Loading HuggingFace sentiment model...")
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    truncation=True,
    max_length=512
)

print("Loading zero-shot topic classifier...")
topic_model = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

STOP_WORDS = set(stopwords.words("english"))
print("✔ All models loaded\n")

# =============================================================================
#  PART 1 — LOAD CSV DATASET
#  Following assignment tip: "skip scraping — load CSV directly"
# =============================================================================

Loading spaCy NER model...
Loading HuggingFace sentiment model...


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✔ All models loaded



### Load crimereport.csv

In [6]:
print("=" * 55)
print("  PART 1 — LOAD DATASET")
print("=" * 55)

if not os.path.exists(TXT_PATH):
    print(f"❌ Dataset not found at: {TXT_PATH}")
    print("\n  Please follow these steps:")
    print("  1. Go to kaggle.com/datasets/cameliasiadat/crimereport")
    print("  2. Download crimereport.txt")
    print("  3. Upload to Google Drive at:")
    print("     MyDrive/AI_DATASETS/TEXT_AI/CrimeReport (1).txt")
    print("  4. Re-run this cell")
    raise FileNotFoundError("crimereport.txt not found — upload it to Drive first!")

# Auto-detect separator — txt files can be comma or tab separated
try:
    df = pd.read_csv(TXT_PATH, sep=',')
    if df.shape[1] == 1:  # Only 1 column means wrong separator
        print("  Comma separator failed — trying tab separator...")
        df = pd.read_csv(TXT_PATH, sep='\t')
except Exception:
    df = pd.read_csv(TXT_PATH, sep='\t')

print(f"✔ Loaded crimereport.txt")
print(f"  Total rows : {len(df)}")
print(f"  Columns    : {list(df.columns)}")
display(df.head(3))
df = df.head(MAX_ROWS)
print(f"\n  Processing : first {MAX_ROWS} rows")

# Identify text column
text_col = None
for col in ["description", "text", "report", "crime_description",
            "incident", "details", "content", "body", "narrative"]:
    if col in df.columns:
        text_col = col
        break
if text_col is None:
    text_col = df.columns[0]

# Identify source column
source_col = None
for col in ["source", "platform", "type", "category", "media"]:
    if col in df.columns:
        source_col = col
        break

print(f"\n  Text column   : '{text_col}'")
print(f"  Source column : '{source_col or 'Not found — using default'}'")
print(f"\n  Sample record :")
print(f"  {str(df[text_col].iloc[0])[:150]}...")
print(f"\n✔ PART 1 COMPLETE — {len(df)} records ready")

# =============================================================================
#  PART 2 — TEXT PREPROCESSING
#  Clean, normalize, tokenize using NLTK as required by assignment
# =============================================================================

  PART 1 — LOAD DATASET
✔ Loaded crimereport.txt
  Total rows : 114
  Columns    : ['{"lang": "en", "favorited": false, "truncated": false, "text": "Active crime scene on I-59/20 near Jeff/Tusc Co line. One dead, one injured; shooting involved. Police search in the area; traffic stopped", "created_at": "Fri Jan 31 05:51:59 +0000 2014", "retweeted": false, "source": "<a href=\\"http://tapbots.com/software/tweetbot/mac\\" rel=\\"nofollow\\">Tweetbot for Mac</a>", "place": {"country_code": "US", "url": "https://api.twitter.com/1.1/geo/id/cf44347a08102884.json", "country": "United States", "place_type": "city", "bounding_box": {"type": "Polygon", "coordinates": [[[-86.926154, 33.267324], [-86.598948, 33.267324], [-86.598948, 33.471006], [-86.926154, 33.471006]]]}, "contained_within": [], "full_name": "Hoover, AL", "attributes": {}, "id": "cf44347a08102884", "name": "Hoover"}, "user": {"id": 15220806, "profile_sidebar_fill_color": "DDEEF6", "profile_text_color": "333333", "followers_count":

,"{""lang"": ""en"", ""favorited"": false, ""truncated"": false, ""text"": ""Active crime scene on I-59/20 near Jeff/Tusc Co line. One dead, one injured; shooting involved. Police search in the area; traffic stopped"", ""created_at"": ""Fri Jan 31 05:51:59 +0000 2014"", ""retweeted"": false, ""source"": ""<a href=\""http://tapbots.com/software/tweetbot/mac\"" rel=\""nofollow\"">Tweetbot for Mac</a>"", ""place"": {""country_code"": ""US"", ""url"": ""https://api.twitter.com/1.1/geo/id/cf44347a08102884.json"", ""country"": ""United States"", ""place_type"": ""city"", ""bounding_box"": {""type"": ""Polygon"", ""coordinates"": [[[-86.926154, 33.267324], [-86.598948, 33.267324], [-86.598948, 33.471006], [-86.926154, 33.471006]]]}, ""contained_within"": [], ""full_name"": ""Hoover, AL"", ""attributes"": {}, ""id"": ""cf44347a08102884"", ""name"": ""Hoover""}, ""user"": {""id"": 15220806, ""profile_sidebar_fill_color"": ""DDEEF6"", ""profile_text_color"": ""333333"", ""followers_count"": 118021, ""location"": ""Alabama"", ""profile_background_color"": ""C0DEED"", ""listed_count"": 1705, ""utc_offset"": -21600, ""statuses_count"": 76381, ""description"": ""Media meteorologist. WeatherBrains host. Weather geek."", ""friends_count"": 52014, ""profile_link_color"": ""0084B4"", ""profile_image_url"": ""https://pbs.twimg.com/profile_images/1890149584/spannwantsyou_normal.jpg"", ""geo_enabled"": true, ""profile_banner_url"": ""https://pbs.twimg.com/profile_banners/15220806/1381811159"", ""profile_background_image_url"": ""http://abs.twimg.com/images/themes/theme1/bg.png"", ""screen_name"": ""spann"", ""lang"": ""en"", ""profile_background_tile"": false, ""favourites_count"": 27, ""name"": ""James Spann"", ""url"": ""http://t.co/7bmABA89RQ"", ""created_at"": ""Tue Jun 24 16:02:10 +0000 2008"", ""time_zone"": ""Central Time (US & Canada)"", ""protected"": false}, ""retweet_count"": 66, ""id"": 429129916446031872, ""favorite_count"": 4}"
0,"{""lang"": ""en"", ""favorited"": false, ""truncated""..."
1,"{""lang"": ""en"", ""favorited"": false, ""truncated""..."
2,"{""lang"": ""en"", ""favorited"": false, ""truncated""..."



  Processing : first 200 rows

  Text column   : '{"lang": "en", "favorited": false, "truncated": false, "text": "Active crime scene on I-59/20 near Jeff/Tusc Co line. One dead, one injured; shooting involved. Police search in the area; traffic stopped", "created_at": "Fri Jan 31 05:51:59 +0000 2014", "retweeted": false, "source": "<a href=\"http://tapbots.com/software/tweetbot/mac\" rel=\"nofollow\">Tweetbot for Mac</a>", "place": {"country_code": "US", "url": "https://api.twitter.com/1.1/geo/id/cf44347a08102884.json", "country": "United States", "place_type": "city", "bounding_box": {"type": "Polygon", "coordinates": [[[-86.926154, 33.267324], [-86.598948, 33.267324], [-86.598948, 33.471006], [-86.926154, 33.471006]]]}, "contained_within": [], "full_name": "Hoover, AL", "attributes": {}, "id": "cf44347a08102884", "name": "Hoover"}, "user": {"id": 15220806, "profile_sidebar_fill_color": "DDEEF6", "profile_text_color": "333333", "followers_count": 118021, "location": "Alabama", "profi

### Text Preprocessing Functions

In [7]:
print("\n" + "=" * 55)
print("  PART 2 — TEXT PREPROCESSING (NLTK)")
print("=" * 55)

def clean_text(text):
    """
    Remove noise and normalize text:
    - Remove URLs, mentions, hashtags
    - Remove special characters
    - Lowercase and normalize whitespace
    """
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)      # Remove URLs
    text = re.sub(r"@\w+|#\w+", "", text)           # Remove mentions/hashtags
    text = re.sub(r"[^a-z0-9\s.,!?'-]", "", text)  # Keep basic punctuation
    text = re.sub(r"\s+", " ", text).strip()         # Normalize whitespace
    return text


def tokenize_and_clean(text):
    """
    Tokenize using NLTK and remove stopwords.
    Returns list of meaningful tokens.
    """
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t.isalpha() and t not in STOP_WORDS]
    return tokens


print("✔ Preprocessing functions defined (NLTK)")

# =============================================================================
#  PART 3 — NLP ANALYSIS
#  NER + Sentiment + Topic Classification
# =============================================================================


  PART 2 — TEXT PREPROCESSING (NLTK)
✔ Preprocessing functions defined (NLTK)


### NER — Named Entity Recognition

In [8]:
print("\n" + "=" * 55)
print("  PART 3 — NLP ANALYSIS")
print("=" * 55)

def extract_entities(text):
    """
    spaCy NER to extract:
    - PERSON   → people / names
    - GPE/LOC  → locations
    - ORG      → organizations
    - DATE     → dates
    Returns formatted string of all entities found.
    """
    doc      = nlp(text[:1000])
    entities = {}
    for ent in doc.ents:
        if ent.label_ in ("PERSON", "GPE", "LOC", "ORG", "DATE"):
            entities.setdefault(ent.label_, []).append(ent.text)

    parts = []
    for label, values in entities.items():
        unique = list(dict.fromkeys(values))[:3]
        parts.append(f"{label}: {', '.join(unique)}")
    return " | ".join(parts) if parts else "None"


def extract_location(text):
    """Return first GPE or LOC entity as primary location."""
    doc  = nlp(text[:1000])
    locs = [e.text for e in doc.ents if e.label_ in ("GPE", "LOC")]
    return locs[0] if locs else "Unknown"


print("✔ NER functions defined (spaCy)")


  PART 3 — NLP ANALYSIS
✔ NER functions defined (spaCy)


### Crime Type Extraction

In [9]:
CRIME_MAP = {
    "Robbery":      ["robbery", "rob", "mugged", "held up", "armed robbery"],
    "Theft":        ["theft", "stolen", "shoplifting", "pickpocket", "stole", "burglar"],
    "Assault":      ["assault", "attack", "battery", "stabbing", "shooting", "shot"],
    "Fire / Arson": ["fire", "arson", "blaze", "burned", "flames"],
    "Drug Offense": ["drug", "narcotic", "cocaine", "heroin", "meth", "seized"],
    "Vandalism":    ["vandal", "graffiti", "damage", "broken", "smashed"],
    "Accident":     ["accident", "crash", "collision", "wreck"],
    "Disturbance":  ["disturbance", "fight", "altercation", "riot", "protest"],
    "Murder":       ["murder", "homicide", "killed", "dead body"],
    "Missing":      ["missing", "disappeared", "lost", "abducted"],
}

def extract_crime_type(text):
    """Extract crime type using keyword matching."""
    text_lower = text.lower()
    for crime, keywords in CRIME_MAP.items():
        if any(kw in text_lower for kw in keywords):
            return crime
    return "Unknown"

print("✔ Crime type extraction defined")

✔ Crime type extraction defined


### Sentiment Analysis

In [10]:
def get_sentiment(text):
    """
    HuggingFace sentiment analysis on preprocessed text.
    Returns 'Positive', 'Negative', or 'Neutral'.
    """
    result = sentiment_model(text[:512])[0]
    label  = result["label"]
    score  = result["score"]
    if label == "NEGATIVE" and score > 0.6:
        return "Negative"
    elif label == "POSITIVE" and score > 0.6:
        return "Positive"
    return "Neutral"

print("✔ Sentiment analysis defined (HuggingFace)")

✔ Sentiment analysis defined (HuggingFace)


### Topic Classification

In [11]:
TOPIC_LABELS = [
    "Theft / Robbery",
    "Physical Assault",
    "Road Accident / Vehicle Incident",
    "Fire / Arson",
    "Drug Related",
    "Public Disturbance",
    "Murder / Homicide",
    "Vandalism / Property Damage",
    "Missing Person",
    "Other Crime",
]

def classify_topic(text):
    """Zero-shot topic classification using HuggingFace BART."""
    result = topic_model(text[:512], candidate_labels=TOPIC_LABELS)
    return result["labels"][0]

print("✔ Topic classification defined (HuggingFace zero-shot)")

✔ Topic classification defined (HuggingFace zero-shot)


### Severity Classification

In [12]:
SEVERITY_HIGH   = ["murder", "homicide", "shooting", "stabbing", "fire",
                    "arson", "assault", "rape", "kidnap", "explosion"]
SEVERITY_MEDIUM = ["robbery", "theft", "stolen", "drug", "burglary",
                    "accident", "crash", "fight", "fraud"]
SEVERITY_LOW    = ["vandalism", "disturbance", "noise", "missing", "shoplifting"]

def get_severity(text):
    """Classify severity as High / Medium / Low."""
    text_lower = text.lower()
    if any(kw in text_lower for kw in SEVERITY_HIGH):
        return "High"
    if any(kw in text_lower for kw in SEVERITY_MEDIUM):
        return "Medium"
    return "Low"

print("✔ Severity classification defined")

✔ Severity classification defined


### Process All Records

In [13]:
print(f"\nRunning NLP on all {len(df)} records...\n")

records = []

for i, row in df.iterrows():
    text_id = f"TXT_{i+1:03d}"
    raw     = str(row[text_col]).strip()
    source  = str(row[source_col]) if source_col else "CrimeReport / Kaggle"

    if not raw or raw == "nan":
        continue

    # PART 2 — Preprocess
    cleaned = clean_text(raw)
    tokens  = tokenize_and_clean(cleaned)

    # PART 3 — NLP Analysis
    crime_type = extract_crime_type(raw)
    entities   = extract_entities(raw)
    location   = extract_location(raw)
    sentiment  = get_sentiment(cleaned)
    topic      = classify_topic(cleaned)
    severity   = get_severity(raw)

    # Excerpt for output table
    excerpt = (raw[:150] + "...") if len(raw) > 150 else raw

    records.append({
        "Text_ID":         text_id,
        "Source":          source,
        "Raw_Text":        excerpt,
        "Crime_Type":      crime_type,
        "Location_Entity": location,
        "Sentiment":       sentiment,
        "Entities":        entities,
        "Topic":           topic,
        "Severity_Label":  severity,
        "Tokens":          ", ".join(tokens[:10]),
    })

    if (i + 1) % 20 == 0:
        print(f"  ✔ {i+1} / {len(df)} processed...")

print(f"\n✔ Done! {len(records)} records processed.")


Running NLP on all 114 records...



You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  ✔ 20 / 114 processed...
  ✔ 40 / 114 processed...
  ✔ 60 / 114 processed...
  ✔ 80 / 114 processed...
  ✔ 100 / 114 processed...

✔ Done! 114 records processed.


### Save and Display Output

In [14]:
output_df = pd.DataFrame(records)

# Save full output with all fields
output_df.to_csv(OUTPUT_CSV.replace(".csv", "_full.csv"), index=False)

# Assignment output — exact columns
assignment_df = output_df[["Text_ID", "Crime_Type", "Location_Entity",
                             "Sentiment", "Topic", "Severity_Label"]].copy()
assignment_df.to_csv(OUTPUT_CSV, index=False)

print(f"\n✔ text_output.csv      — {len(assignment_df)} records (submit this)")
print(f"✔ text_full_output.csv — includes entities and tokens")

print(f"\n{'='*55}")
print("  STRUCTURED OUTPUT TABLE (first 10 rows)")
print(f"{'='*55}")
display(assignment_df.head(10))


✔ text_output.csv      — 114 records (submit this)
✔ text_full_output.csv — includes entities and tokens

  STRUCTURED OUTPUT TABLE (first 10 rows)


,Text_ID,Crime_Type,Location_Entity,Sentiment,Topic,Severity_Label
0,TXT_001,Assault,Rocky Mount,Negative,Public Disturbance,High
1,TXT_002,Unknown,Chicago,Negative,Public Disturbance,Low
2,TXT_003,Assault,Unknown,Negative,Public Disturbance,Low
3,TXT_004,Murder,London,Negative,Murder / Homicide,High
4,TXT_005,Unknown,crimes\u2019 http://t.co/SLuXLgbQbU,Negative,Public Disturbance,Low
5,TXT_006,Robbery,Buffalo,Negative,Theft / Robbery,Medium
6,TXT_007,Unknown,Pakistan,Negative,Public Disturbance,Low
7,TXT_008,Unknown,NRA,Negative,Public Disturbance,Low
8,TXT_009,Drug Offense,Baltimore,Negative,Drug Related,Medium
9,TXT_010,Unknown,England,Negative,Public Disturbance,Low


### Verify Output Columns

In [15]:
print(f"\n{'='*55}")
print("  OUTPUT COLUMNS CHECK")
print(f"{'='*55}")
expected = ["Text_ID", "Crime_Type", "Location_Entity",
            "Sentiment", "Topic", "Severity_Label"]
for col in expected:
    status = "✔" if col in assignment_df.columns else "❌"
    print(f"  {status} {col}")

print(f"\n{'='*55}")
print("  SENTIMENT DISTRIBUTION")
print(f"{'='*55}")
display(assignment_df["Sentiment"].value_counts().reset_index().rename(
    columns={"index": "Sentiment", "Sentiment": "Count"}
))

print(f"\n{'='*55}")
print("  SEVERITY DISTRIBUTION")
print(f"{'='*55}")
display(assignment_df["Severity_Label"].value_counts().reset_index().rename(
    columns={"index": "Severity_Label", "Severity_Label": "Count"}
))

print(f"\n✔ Download text_output.csv from your Google Drive")
print(f"  Share with your team for the integration step.")


  OUTPUT COLUMNS CHECK
  ✔ Text_ID
  ✔ Crime_Type
  ✔ Location_Entity
  ✔ Sentiment
  ✔ Topic
  ✔ Severity_Label

  SENTIMENT DISTRIBUTION


,Count,count
0,Negative,112
1,Positive,2



  SEVERITY DISTRIBUTION


,Count,count
0,Low,84
1,High,20
2,Medium,10



✔ Download text_output.csv from your Google Drive
  Share with your team for the integration step.
